# Feature Engineering for Demand Forecasting

This notebook builds temporal, rolling, lag, and calendar features from `data/processed/sales_clean.parquet` to prepare input datasets for predictive modeling.

### Engineered Feature Categories:
1. **Lag Features**: `lag_1`, `lag_2`, `lag_4`, `lag_51`, `lag_52` (justified by ACF analysis in Step 3).
2. **Rolling Window Features**: `rolling_mean_4`, `rolling_std_4`, `rolling_mean_12` (strictly shifted by 1 week prior to rolling calculations to eliminate data leakage).
3. **Calendar & Cyclical Features**: `month`, `week_of_year`, `day_of_week`, along with sine/cosine cyclical transformations (`week_sin`, `week_cos`).
4. **Flags & Availability**: `is_promo_event` and `markdown_available` (flagging records $\ge$ Nov 1, 2011 to delineate data collection availability).
5. **Store Level Attributes**: `store_type`, `store_size`.


## 0. Environment Setup & Data Ingestion

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path('../data/processed/sales_clean.parquet')
if not data_path.exists():
    data_path = Path('data/processed/sales_clean.parquet')

df = pd.read_parquet(data_path)
df['date'] = pd.to_datetime(df['date'])

# Crucial: Sort deterministically by store, department, and date
df = df.sort_values(['store_id', 'dept_id', 'date']).reset_index(drop=True)

print(f"Loaded {len(df):,} cleaned sales records across {len(df[['store_id', 'dept_id']].drop_duplicates()):,} store-department time series.")


Loaded 421,570 cleaned sales records across 3,331 store-department time series.


## 1. Lag Feature Creation (ACF Driven)

Lags `1`, `2`, `4`, `51`, and `52` are created per `(store_id, dept_id)` group.
- `lag_1` & `lag_2` capture short-term weekly momentum.
- `lag_51` & `lag_52` capture annual seasonal alignment (prior year holiday weeks).


In [2]:
grouped = df.groupby(['store_id', 'dept_id'])['weekly_sales']

for k in [1, 2, 4, 51, 52]:
    df[f'lag_{k}'] = grouped.shift(k)

print("Lag features created successfully.")


Lag features created successfully.


## 2. Rolling Window Features (Strict Leakage Prevention)

> **Leakage Prevention**: All rolling windows (`rolling_mean_4`, `rolling_std_4`, `rolling_mean_12`) are calculated by **shifting `weekly_sales` by 1 week prior to applying rolling functions**. This ensures that week $t$'s feature vector contains only historical information from weeks $< t$.


In [3]:
for w in [4, 12]:
    df[f'rolling_mean_{w}'] = df.groupby(['store_id', 'dept_id'])['weekly_sales'].transform(lambda s: s.shift(1).rolling(w).mean())

df['rolling_std_4'] = df.groupby(['store_id', 'dept_id'])['weekly_sales'].transform(lambda s: s.shift(1).rolling(4).std())

print("Rolling features created successfully.")


Rolling features created successfully.


## 3. Calendar & Cyclical Encoding

In [4]:
# Continuous sine/cosine encoding of week_of_year for seasonal continuity
df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52.0)
df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52.0)

print("Cyclical calendar encodings (week_sin, week_cos) generated.")


Cyclical calendar encodings (week_sin, week_cos) generated.


## 4. Promo & Data Availability Flags

In [5]:
# Explicit flag to separate pre-Nov-2011 missing markdown tracking from genuine zero markdowns
df['markdown_available'] = (df['date'] >= '2011-11-01').astype(bool)

print("Data availability flag `markdown_available` created.")


Data availability flag `markdown_available` created.


## 5. Verification & Spot-Check

In [6]:
# 1. Total row count and column list
print(f"=== Dataset Shape & Columns ===")
print(f"Total Rows: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print(f"Column Names: {list(df.columns)}")

# 2. NaN Percentage in Engineered Features
print(f"\n=== Missing Data (NaN %) in Engineered Features ===")
feature_cols = ['lag_1', 'lag_2', 'lag_4', 'lag_51', 'lag_52', 'rolling_mean_4', 'rolling_std_4', 'rolling_mean_12']
num_series = len(df[['store_id', 'dept_id']].drop_duplicates())

for col in feature_cols:
    nan_count = df[col].isna().sum()
    nan_pct = (nan_count / len(df)) * 100
    print(f"  {col:16s} : {nan_count:7,} NaNs ({nan_pct:5.2f}%) | Expectation: ~{int(col.split('_')[-1]) * num_series:,}")

# 3. Manual Spot-Check on Store 1, Department 1
print(f"\n=== Manual Spot-Check: Store 1, Department 1 ===")
spot_check = (
    df[(df['store_id'] == 1) & (df['dept_id'] == 1)]
    .sort_values('date')
    [['date', 'weekly_sales', 'lag_1', 'lag_2', 'rolling_mean_4', 'rolling_std_4']]
    .head(6)
)
display(spot_check)


=== Dataset Shape & Columns ===
Total Rows: 421,570
Total Columns: 30
Column Names: ['store_id', 'dept_id', 'date', 'weekly_sales', 'temperature', 'fuel_price', 'cpi', 'unemployment', 'markdown_total', 'store_type', 'store_size', 'year', 'month', 'week_of_year', 'day_of_week', 'is_promo_event', 'year_month', 'sales_zscore', 'is_outlier', 'lag_1', 'lag_2', 'lag_4', 'lag_51', 'lag_52', 'rolling_mean_4', 'rolling_mean_12', 'rolling_std_4', 'week_sin', 'week_cos', 'markdown_available']

=== Missing Data (NaN %) in Engineered Features ===
  lag_1            :   3,331 NaNs ( 0.79%) | Expectation: ~3,331
  lag_2            :   6,625 NaNs ( 1.57%) | Expectation: ~6,662
  lag_4            :  13,134 NaNs ( 3.12%) | Expectation: ~13,324
  lag_51           : 157,496 NaNs (37.36%) | Expectation: ~169,881
  lag_52           : 160,487 NaNs (38.07%) | Expectation: ~173,212
  rolling_mean_4   :  13,134 NaNs ( 3.12%) | Expectation: ~13,324
  rolling_std_4    :  13,134 NaNs ( 3.12%) | Expectation: ~13,32

,date,weekly_sales,lag_1,lag_2,rolling_mean_4,rolling_std_4
0,2010-02-05,24924.50,NaN,NaN,NaN,NaN
1,2010-02-12,46039.49,24924.50,NaN,NaN,NaN
2,2010-02-19,41595.55,46039.49,24924.50,NaN,NaN
3,2010-02-26,19403.54,41595.55,46039.49,NaN,NaN
4,2010-03-05,21827.90,19403.54,41595.55,32990.77,12832.106391
5,2010-03-12,21043.39,21827.90,19403.54,32216.62,13554.047185


## 6. Export Feature Dataset

In [7]:
output_dir = Path('../data/processed')
if not output_dir.parent.exists():
    output_dir = Path('data/processed')

output_dir.mkdir(parents=True, exist_ok=True)
parquet_out = output_dir / 'sales_features.parquet'

df.to_parquet(parquet_out, index=False)

print("=== Feature Dataset Export Confirmation ===")
print(f"Saved Parquet File: {parquet_out.resolve()}")
print(f"File Size: {parquet_out.stat().st_size / 1e6:.2f} MB")
print(f"Total Rows Preserved: {len(df):,}")


=== Feature Dataset Export Confirmation ===
Saved Parquet File: D:\Ssshhhh\Projexts\Supply Chain\supply-chain-demand-forecasting\data\processed\sales_features.parquet
File Size: 26.45 MB
Total Rows Preserved: 421,570
